# CoffeeFG-YOLO26 v2 — A0 real validation

Notebook ini hanya memakai A0 nyata. Tahap 1 melatih D0 (P3–P5) dan D1 (P2–P5), lalu diagnostic memilih pasangan refiner. Test tidak diekstrak dan tidak dibuka.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ['PYTHONPATH'] = str(SRC) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO)
import coffee_detector
print('REPO:', REPO)
print('IMPORT:', coffee_detector.__file__)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
preferred = [
    DRIVE / 'Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar',
    DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-bundle/A0_real.tar',
    DRIVE / '02_RISET_DAN_PROYEK/Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar',
]
A0_ARCHIVE = next((path for path in preferred if path.is_file()), None)
if A0_ARCHIVE is None:
    matches = sorted(path for path in DRIVE.rglob('A0_real.tar') if path.is_file())
    if not matches:
        shortcut_root = Path('/content/drive/.shortcut-targets-by-id')
        if shortcut_root.is_dir():
            matches = sorted(path for path in shortcut_root.rglob('A0_real.tar') if path.is_file())
    assert matches, 'A0_real.tar tidak ditemukan di My Drive atau shortcut proyek.'
    A0_ARCHIVE = matches[0]
PROJECT_DRIVE = DRIVE / 'Coffee_Bean_Detection'
OUTPUT_ROOT = PROJECT_DRIVE / 'experiments/coffee-fg-v2'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('A0 ARCHIVE:', A0_ARCHIVE)
print('OUTPUT:', OUTPUT_ROOT)

In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_development

DATA_ROOT = restore_real_a0_development(A0_ARCHIVE, '/content/sni21-a0-development')
assert (DATA_ROOT / 'train/images').is_dir()
assert (DATA_ROOT / 'val/images').is_dir()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh diekstrak.'
print('DATASET TRAIN+VAL SIAP:', DATA_ROOT)

## Tahap 1 — baseline P3 versus P2

D0 dan D1 memakai seed 42, 50 epoch maksimum, validation saja, dan output langsung ke Drive. Jika runtime terputus, jalankan ulang notebook; runner akan resume dari `last.pt`.

In [ ]:
RUN_STAGE1 = True
if RUN_STAGE1:
    command = [
        sys.executable, '-u', '-m', 'coffee_detector.experiments.run_coffee_fg_screening',
        '--data-root', str(DATA_ROOT),
        '--output-root', str(OUTPUT_ROOT),
        '--models', 'D0', 'D1',
        '--seeds', '42',
        '--evaluation-split', 'val',
        '--device', '0',
    ]
    print('MENJALANKAN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)
else:
    print('Tahap 1 dilewati.')

## Diagnostic wajib

Diagnostic menguji apakah P2 memperbaiki akses proposal dan apakah masih ada ruang kesalahan klasifikasi setelah objek berhasil dilokalisasi.

In [ ]:
DIAGNOSTIC = OUTPUT_ROOT / 'val_reports/diagnostic_seed42.json'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.coffee_fg_diagnostics',
    '--p3-checkpoint', str(OUTPUT_ROOT / 'D0_seed42/weights/best.pt'),
    '--p2-checkpoint', str(OUTPUT_ROOT / 'D1_seed42/weights/best.pt'),
    '--data-root', str(DATA_ROOT),
    '--output', str(DIAGNOSTIC),
    '--split', 'val',
    '--candidate-counts', '50', '100', '300', '500',
    '--max-det', '500',
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import json
decision = json.loads(DIAGNOSTIC.read_text(encoding='utf-8'))['decision']
print(json.dumps(decision, indent=2, ensure_ascii=False))
if not decision['classification_refinement_rational']:
    print('STOP: refiner fine-grained tidak rasional pada A0 validation.')
else:
    print('PASANGAN YANG BOLEH DILATIH:', decision['recommended_refiners'])

## Tahap 2 — refiner terpilih (manual)

Ubah `RUN_REFINER=True` hanya apabila diagnostic menyatakan refinement rasional. Notebook mengambil pasangan R0/R1 atau R2/R3 secara otomatis.

In [ ]:
RUN_REFINER = False
if RUN_REFINER:
    assert decision['classification_refinement_rational'], 'Gate diagnostic gagal.'
    refiners = list(decision['recommended_refiners'])
    command = [
        sys.executable, '-u', '-m', 'coffee_detector.experiments.run_coffee_fg_screening',
        '--data-root', str(DATA_ROOT),
        '--output-root', str(OUTPUT_ROOT),
        '--models', *refiners,
        '--seeds', '42',
        '--evaluation-split', 'val',
        '--diagnostic-report', str(DIAGNOSTIC),
        '--device', '0',
    ]
    print('MENJALANKAN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)
else:
    print('Refiner belum dijalankan. Kirim hasil diagnostic terlebih dahulu.')